# Notebook de definición del problema y entorno reproducible — Fase 1

**Proyecto:** Análisis de alojamientos Airbnb en Santiago de Chile para la
decisión de inversión inmobiliaria

---

### ¿Qué produce la Fase 1?

A la Fase 2 le llega un conjunto de datos y lo deja listo para el análisis.
Para que eso funcione, antes alguien tuvo que decidir **qué problema se
resuelve, con qué datos y sobre qué entorno**. Ese es el trabajo de esta fase,
y el orden en que va el cuaderno es:

**Definir → Configurar → Verificar → Estructurar → Documentar → Versionar**

Cada paso queda implementado como una **función** reutilizable y termina en una
**comprobación con evidencia**. La diferencia con la Fase 2 es el objeto de
trabajo: alla se transforman datos, acá se arma y se verifica el andamiaje que
hace que esa transformación la pueda reproducir otra persona.

> **Lo que esta fase NO hace.** No limpia, no imputa, no codifica ni escala. Si
> en este cuaderno aparece una imputación, el trabajo quedo adelantado de fase
> y la rúbrica de la Fase 2 se queda sin evidencia propia.

## Introducción

Un inversionista que está pensando en comprar un departamento para arriendo
turístico en Santiago no tiene evidencia sistemática sobre qué ubicación,
tipología y equipamiento se asocian a mayores ingresos. El proyecto describe
esa relación a partir de los alojamientos publicados en Airbnb (Inside Airbnb,
captura del 25–26 de septiembre de 2026).

**Objetivo general (Fase 1).** Definir la problemática y los objetivos del
proyecto, y dejar funcionando un entorno de trabajo reproducible y versionado
que sostenga las fases que vienen.

**Objetivos específicos.**
- Declarar el problema, las preguntas que se van a responder y los criterios de éxito.
- Verificar que el entorno de ejecución sea el del proyecto y no el del sistema.
- Construir la estructura del repositorio y los artefactos que la hacen reproducible.
- Documentar la procedencia del conjunto de datos y evaluarla frente a los criterios del curso.
- Dejar trazabilidad entre el mapa conceptual, el repositorio y este cuaderno.

**Entregables de la fase:** informe técnico, repositorio con estructura y
*commits* documentados, `README` técnico, este cuaderno ejecutado y el mapa
conceptual de la Formativa 1.

### Herramientas del ecosistema científico

| Pieza | Para qué la usamos | Qué falla si se omite |
|---|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto. | Un paquete «desaparece» o el compañero no logra reproducir. |
| JupyterLab + `ipykernel` | Ejecutar el cuaderno con el intérprete del proyecto. | `ModuleNotFoundError` con un paquete que sí está instalado. |
| `requirements.txt` | Declarar las dependencias exactas. | «A mí me funciona» y a nadie mas. |
| Git | Registrar la historia del trabajo y su autoría. | No queda evidencia de la contribución de cada uno. |
| GitHub | Colaborar y publicar la evidencia evaluada. | El repositorio termina siendo un depósito de archivos. |
| `pathlib` | Rutas relativas independientes del sistema operativo. | `C:/Users/...` funciona en un solo computador del mundo. |

En esta fase se usan sobre todo módulos de la biblioteca estándar, porque el
cuaderno construye y verifica el entorno y todavía no analiza datos. `numpy` y
`pandas` salen únicamente para comprobar que están disponibles y para escribir
las tablas de documentación.

In [1]:
import sys                        # el intérprete en uso: se verifica mas abajo
import json                       # exportar metadatos legibles
import platform                   # el sistema operativo, para la bitácora
import subprocess                 # consultas a Git desde el cuaderno
import shutil                     # ubicar ejecutables (git) sin asumir que existen
import importlib                  # importar el módulo que este cuaderno escribe
import importlib.util             # comprobar si un paquete está instalado
from pathlib import Path          # rutas que sirven igual en cualquier sistema
from datetime import date

# Revisión del entorno: instala en el kernel lo que haga falta.
print("Intérprete del kernel:", sys.executable)

REQUERIDOS = {"numpy": "numpy", "pandas": "pandas"}   # modulo: paquete pip
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                # acá solo para verificar el entorno
import pandas as pd               # las tablas de documentación de la fase

print("numpy", np.__version__, "· pandas", pd.__version__)

# Reproducibilidad: la misma semilla del cuaderno de la Fase 2 y del
# muestreo estratificado de src/generar_muestra.py.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 ·", date.today().isoformat())

Intérprete del kernel: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/.venv/bin/python3
numpy 2.5.4 · pandas 3.1.1
Cuaderno de la Fase 1 · 2026-11-10


## 1. Definición del problema

**Qué hace este paso.** Deja el problema escrito como una **estructura de
datos**, no como un párrafo suelto. Todo lo que declara el informe
—problemática, objetivos, alcance, criterios de éxito— queda acá en un
diccionario del que después salen el `README`, los metadatos y la tabla de
vinculación con el mapa conceptual.

**Por qué así.** Un dato escrito una sola vez no se puede contradecir consigo
mismo. Si el título del proyecto cambia, se cambia en un solo lugar y se
propaga a todos los artefactos. Es el mismo principio que en la Fase 2 lleva a
encapsular cada paso en una función en vez de copiar y pegar.

In [2]:
PROYECTO = {
    "titulo": "Análisis de alojamientos Airbnb en Santiago de Chile para la decisión de inversión inmobiliaria",
    "grupo": "Grupo 4",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": [
        "Alfredo Valenzuela Aburto",
        "Mauricio Martínez Quiñones",
        "María Laura Lanosa",
    ],
    "problematica": (
        "Un inversionista que está evaluando comprar un departamento para arriendo "
        "turístico en Santiago no tiene evidencia sistemática sobre qué "
        "ubicación, tipología y equipamiento se asocian a mayores ingresos. El "
        "proyecto describe esa relación a partir de 18.904 alojamientos "
        "publicados en Airbnb (Inside Airbnb, captura del 25–26 de septiembre "
        "de 2026), de los que se toma una muestra estratificada por comuna."
    ),
    "pregunta_investigacion": (
        "¿Qué características de ubicación, tipología, capacidad y servicios "
        "presentan los departamentos completos activos en Airbnb Santiago con "
        "mayor ocupación e ingresos estimados?"
    ),
    "objetivo_general": (
        "Describir la relación entre ubicación, configuración y comodidades "
        "de los departamentos completos activos en Airbnb Santiago y su "
        "desempeño comercial estimado (ocupación e ingresos), para identificar "
        "perfiles que orienten una decisión de compra."
    ),
    "objetivos_especificos": [
        "Definir el problema, validar el conjunto y configurar un entorno reproducible y versionado (F1).",
        "Construir el pipeline de carga, anonimización, limpieza y transformación de datos (F2).",
        "Calcular indicadores de desempeño por comuna, configuración y comodidad (F3).",
        "Comunicar los perfiles hallados mediante visualizaciones e informe técnico (F4).",
    ],
    "preguntas": [
        "¿Cómo se distribuyen el precio por noche y la ocupación estimada entre las 32 comunas?",
        "¿Cómo varían el precio y la ocupación estimada según el número de dormitorios, baños y capacidad?",
        "¿Qué comodidades son más frecuentes entre los departamentos del cuartil superior de ocupación estimada, en comparación con el resto?",
    ],
    "criterios_exito": [
        "El repositorio se clona y los cuadernos corren completos sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": [
            "Departamentos y casas completas (room_type = Entire home/apt)",
            "Alojamientos activos: reseña en los últimos 12 meses y disponibilidad futura",
            "Análisis descriptivo de ocupación estimada, precio e ingresos estimados",
        ],
        "excluye": [
            "Rentabilidad o retorno de la inversión (no hay datos de costos)",
            "Modelos predictivos o causales",
            "Análisis de sentimiento y comparación entre ciudades",
        ],
    },
    "limitaciones": [
        "Sesgo de supervivencia: solo salen los anuncios que seguían publicados en septiembre de 2026; los que cerraron no están.",
        "La ocupación estimada sale de las reseñas y tiene un tope de 255 noches (~70 % del año); no es un registro de reservas.",
        "Los ingresos estimados son ocupación × precio publicado: heredan ambos sesgos y no son una métrica independiente.",
        "Los cuadernos trabajan sobre una muestra estratificada del archivo detallado, no sobre el universo completo.",
    ],
}

La función que sigue presenta el proyecto en pantalla. Recibe el
diccionario como parámetro en vez de leer la variable global: así se puede
reutilizar con la configuración de cualquier grupo y probarla por separado.

In [3]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y el mensaje dice QUE falta, no solo que algo fallo.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 76)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nPregunta de investigación")
    print(config["pregunta_investigacion"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas operativas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

ANÁLISIS DE ALOJAMIENTOS AIRBNB EN SANTIAGO DE CHILE PARA LA DECISIÓN DE INVERSIÓN INMOBILIARIA
MCDI500 · Programación para la Ciencia de Datos · Grupo 4

Problemática
Un inversionista que está evaluando comprar un departamento para arriendo turístico en Santiago no tiene evidencia sistemática sobre qué ubicación, tipología y equipamiento se asocian a mayores ingresos. El proyecto describe esa relación a partir de 18.904 alojamientos publicados en Airbnb (Inside Airbnb, captura del 25–26 de septiembre de 2026), de los que se toma una muestra estratificada por comuna.

Pregunta de investigación
¿Qué características de ubicación, tipología, capacidad y servicios presentan los departamentos completos activos en Airbnb Santiago con mayor ocupación e ingresos estimados?

Objetivo general
Describir la relación entre ubicación, configuración y comodidades de los departamentos completos activos en Airbnb Santiago y su desempeño comercial estimado (ocupación e ingresos), para identificar perfil

El alcance y las limitaciones se presentan aparte porque cumplen otra
función: delimitan lo que **no** se va a hacer. Es lo que evita que el informe
de la Fase 4 sea juzgado por algo que nunca se prometió.

In [4]:
print("Alcance de la Fase 1 y del proyecto")
print("  Incluye:")
for item in PROYECTO["alcance"]["incluye"]:
    print(f"    + {item}")
print("  Excluye:")
for item in PROYECTO["alcance"]["excluye"]:
    print(f"    - {item}")

print("\nSesgos y limitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

Alcance de la Fase 1 y del proyecto
  Incluye:
    + Departamentos y casas completas (room_type = Entire home/apt)
    + Alojamientos activos: reseña en los últimos 12 meses y disponibilidad futura
    + Análisis descriptivo de ocupación estimada, precio e ingresos estimados
  Excluye:
    - Rentabilidad o retorno de la inversión (no hay datos de costos)
    - Modelos predictivos o causales
    - Análisis de sentimiento y comparación entre ciudades

Sesgos y limitaciones declaradas
  · Sesgo de supervivencia: solo salen los anuncios que seguían publicados en septiembre de 2026; los que cerraron no están.
  · La ocupación estimada sale de las reseñas y tiene un tope de 255 noches (~70 % del año); no es un registro de reservas.
  · Los ingresos estimados son ocupación × precio publicado: heredan ambos sesgos y no son una métrica independiente.
  · Los cuadernos trabajan sobre una muestra estratificada del archivo detallado, no sobre el universo completo.


> **Sobre los sesgos.** Declararlos antes de haber analizado nada muestra
> comprensión del problema, no debilidad del trabajo. La captura de Inside
> Airbnb solo muestra los alojamientos que **sobrevivieron** en el mercado
> hasta septiembre de 2026, y su métrica de ocupación es una estimación
> basada en reseñas con un tope de 255 noches. Omitirlo sería vender los
> resultados de la Fase 4 como mas sólidos de lo que son.

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba, con evidencia y no por confianza, que el
cuaderno está corriendo con el intérprete del proyecto y que las librerías
declaradas están disponibles en él.

**Por qué es la primera comprobación del proyecto.** Los dos errores más
frecuentes del módulo —un paquete que «desaparece» y un `ModuleNotFoundError`
con una librería que sí se instaló— no son errores de código: son el entorno o
el kernel equivocados. Verificarlo cuesta tres segundos y ahorra horas de
diagnóstico sobre el archivo que no es.

In [5]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. El intérprete que corre ESTE cuaderno. Si la ruta no trae .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde acá,
    #    no desde donde quedó guardado el .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Las librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se frena el cuaderno: se avisa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)

Intérprete       : /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/.venv/bin/python3
Entorno virtual  : [OK] activo
Carpeta de trabajo: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/F1
Sistema          : Linux 7.0.0-42-generic
Python           : 3.14.4

Librerías declaradas
  [OK]    numpy        2.5.4
  [OK]    pandas       3.1.1


  [OK]    matplotlib   3.11.2


  [OK]    seaborn      0.13.2
  [OK]    sklearn      1.9.1


**Cómo se lee este resultado.** Cada línea corresponde a uno de los
acoplamientos que la guía de Git y GitHub describe en su Parte II:

| Lo que ve | Qué significa | Qué hacer |
| --- | --- | --- |
| El intérprete no contiene `.venv` | El kernel no es el del proyecto | *Kernel → Change Kernel* |
| Una librería aparece como `[FALTA]` | Se instaló fuera del entorno | Active el entorno y reinstale |
| La carpeta de trabajo es `F1` o `F2` | Es normal: Jupyter abre el kernel en la carpeta del cuaderno | Nada: la sección 3 localiza la raíz del proyecto |

> **Recomendación.** Deje esta celda como primera celda de todos sus cuadernos,
> también en las fases que vienen. Responde de una la pregunta «¿el problema
> es mi código o mi entorno?», que es la que mas tiempo consume cuando no se
> contesta a tiempo.

## 3. Estructura del repositorio y artefactos

**Qué hace este paso.** Crea las carpetas del proyecto y escribe desde el
cuaderno los dos archivos que hacen reproducible el entorno: `.gitignore` y
`requirements.txt`.

**Por qué desde el cuaderno.** Porque así quedan derivados de la configuración
declarada en la sección 1 y no de lo que a alguien se le ocurrió escribir a
mano. Es la misma lógica de la Fase 2: el artefacto se genera, se verifica y
queda trazado.

In [6]:
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde quedó guardado el cuaderno, no en
    la carpeta desde donde se lanzó JupyterLab. Como cada cuaderno vive en la
    carpeta de su fase (F1, F2, F3 o F4), cuando la carpeta de trabajo es una de
    esas la raíz es la carpeta de arriba. Si el cuaderno corre desde la raíz, se
    usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()

# Rutas de datos COMUNES a todas las fases: las mismas que usa el
# cuaderno de la Fase 2, para que un mismo archivo le sirva a los dos.
DIR_CRUDO = RAIZ / "data" / "raw"          # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                   # diccionario, fichas y metadatos
DIR_SRC = RAIZ / "src"                     # módulos propios reutilizables

# Una carpeta por fase, para cuadernos e informes.
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_CRUDO, DIR_PROCESADO, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    # parents=True crea las intermedias; exist_ok=True evita el error si ya están,
    # que es lo que deja correr el cuaderno completo de nuevo sin que se rompa.
    carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura creada bajo:", RAIZ.resolve())
# Se saltan las carpetas ocultas (.git, .venv, .ipynb_checkpoints) y las cachés.
for ruta in sorted(p.relative_to(RAIZ) for p in RAIZ.rglob("*") if p.is_dir()):
    if any(parte.startswith(".") or parte == "__pycache__" for parte in ruta.parts):
        continue
    print("  ", ruta.as_posix() + "/")

Estructura creada bajo: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500


   F1/
   F1/notebooks/
   F2/
   F3/
   F4/
   data/
   data/processed/
   data/raw/
   docs/
   src/


> **Decisión que el grupo tomó y documentó.** Los datos son comunes a
> todas las fases (`data/`): una sola carpeta `data/raw` y una sola
> `data/processed`, que es lo que el cuaderno de la Fase 2 espera. El archivo
> detallado de Inside Airbnb pesa unos 50 MB y trae datos personales, así que
> **la descarga original no se versiona**: el script `src/generar_muestra.py`
> produce, con semilla fija, la muestra estratificada por comuna **ya
> anonimizada** (D5) —esa si se versiona en `data/raw/`, porque no trae datos
> personales y pesa menos de 10 MB (D7).

### 3.1 El archivo `.gitignore`

Quedan fuera del repositorio cuatro cosas: lo que se regenera (el entorno
virtual, las cachés), lo que es local de cada equipo (archivos del sistema
operativo y del IDE), lo que es secreto (credenciales) y —en este proyecto—
**las descargas originales** de Inside Airbnb (`*.csv.gz`): traen información
personal de anfitriones y pesan mas de lo razonable. La muestra anonimizada de
`data/raw/` **sí entra al repositorio**. Se escribe **antes** del primer
`git add .`: después ya es tarde, porque el archivo quedó en el historial.

In [7]:
CONTENIDO_GITIGNORE = """# Entorno virtual: se reconstruye con requirements.txt
.venv/

# Caché de Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# IDE
.idea/

# Windows
Thumbs.db
desktop.ini

# macOS
.DS_Store

# Variables de entorno y secretos
.env

# Descargas originales de Inside Airbnb: traen datos personales y pesan
# demasiado para versionar. La muestra de data/raw/ SI se versiona porque
# src/generar_muestra.py la deja anonimizada (ver README).
*.csv.gz

# Datos pesados y modelos
*.ckpt
*.pth
"""

ARCHIVO_GITIGNORE = RAIZ / ".gitignore"
ARCHIVO_GITIGNORE.write_text(CONTENIDO_GITIGNORE, encoding="utf-8")

# Verificación de ida y vuelta: se relee lo que se escribió antes de darlo por bueno.
lineas = [l for l in ARCHIVO_GITIGNORE.read_text(encoding="utf-8").splitlines()
          if l and not l.startswith("#")]
assert ".venv/" in lineas, "El entorno virtual debe estar ignorado."
assert "*.csv.gz" in lineas, "Las descargas originales deben estar ignoradas (D7)."
print(f"{ARCHIVO_GITIGNORE} escrito con {len(lineas)} reglas activas.")

/home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/.gitignore escrito con 12 reglas activas.


> **Atención — los datos.** El `.gitignore` solo excluye las descargas
> originales (`*.csv.gz`), no `data/raw/`. La regla que usamos fue: **lo que
> contiene datos personales no se versiona, lo demás sí**. Como la muestra se
> anonimiza en su generación (D5), `data/raw/listings.csv` entra al
> repositorio junto con `data/processed/`. Quien regenere la muestra desde la
> descarga original obtiene exactamente el mismo archivo anonimizado, porque
> el script trabaja con semilla fija.

### 3.2 El archivo `requirements.txt`

Declarar las dependencias es lo que separa «funciona en mi computador» de un
entorno reproducible. Se escriben con la versión exacta observada en el
entorno, no con la última que halla en el momento de instalar.

In [8]:
def escribir_requirements(ruta, versiones):
    """
    Escribe requirements.txt a partir de las versiones observadas en el entorno.

    Parametros
    ----------
    ruta : Path
        Archivo de salida.
    versiones : dict
        Nombre de importacion -> version detectada (o None si falta).

    Retorna
    -------
    list of str
        Lineas escritas, para verificarlas.

    Lanza
    -----
    ValueError
        Si ninguna dependencia pudo detectarse.
    """
    lineas = []
    for nombre, version in versiones.items():
        if version is None:
            continue  # una librería ausente no se declara: primero toca instalarla
        paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
        lineas.append(f"{paquete}=={version}")

    if not lineas:
        raise ValueError("No se detectó ninguna dependencia instalada.")

    # Se agregan las herramientas del flujo de trabajo: no se importan desde
    # el código pero hacen parte del entorno reproducible.
    lineas += ["jupyterlab", "notebook", "ipykernel"]
    ruta.write_text("\n".join(sorted(lineas)) + "\n", encoding="utf-8")
    return sorted(lineas)


ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
requisitos = escribir_requirements(ARCHIVO_REQUIREMENTS, ENTORNO["versiones"])
print(ARCHIVO_REQUIREMENTS, "\n")
print("\n".join(requisitos))

/home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/requirements.txt 

ipykernel
jupyterlab
matplotlib==3.11.2
notebook
numpy==2.5.4
pandas==3.1.1
scikit-learn==1.9.1
seaborn==0.13.2


> **Atención.** Este archivo se generó a partir de las librerías
> **realmente presentes** en el entorno. `pip freeze > requirements.txt` es la
> alternativa de siempre y también sirve, con una diferencia: mete además las
> dependencias indirectas, lo que da un entorno más fiel pero más difícil de
> leer. Nosotros elegimos la primera opción y lo dejamos documentado acá.

## 4. Módulos: del cuaderno a `src/`

**Qué hace este paso.** Escribe un módulo `src/utilidades.py` y lo importa de
vuelta.

**Por qué.** La Unidad 1 pide funciones, excepciones **y módulos**. Un cuaderno
documenta y comunica; un módulo es lo que permite que la misma función se use
en las cuatro fases sin copiarla. El módulo junta dos utilidades que la Fase 2
va a usar apenas cargue datos: la normalización de nombres de columna y la
seudonimización de identificadores personales (decisión D5).

In [9]:
CODIGO_MODULO = """\
\"\"\"Utilidades compartidas del proyecto MCDI500 - Grupo 4.

Este modulo lo genera el cuaderno de la Fase 1 y lo importan las fases que
vienen. Proyecto: alojamientos Airbnb en Santiago de Chile.
\"\"\"

import re
import hashlib


class ProyectoError(Exception):
    \"\"\"Error propio del proyecto, para distinguirlo de los de las librerias.\"\"\"


def normalizar_nombre(texto):
    \"\"\"Pasa un nombre de columna a minusculas, sin espacios ni signos.

    Ejemplo
    -------
    >>> normalizar_nombre('Bathrooms Text ')
    'bathrooms_text'
    \"\"\"
    if not isinstance(texto, str):
        raise ProyectoError(f\"Se esperaba texto y se recibio {type(texto).__name__}.\")
    limpio = texto.strip().lower()
    limpio = re.sub(r\"[^a-z0-9]+\", \"_\", limpio)   # lo que no sea letra o digito -> _
    return limpio.strip(\"_\")


def normalizar_columnas(nombres):
    \"\"\"Aplica normalizar_nombre a una lista y verifica que no haya duplicados.\"\"\"
    normalizados = [normalizar_nombre(n) for n in nombres]
    if len(set(normalizados)) != len(normalizados):
        raise ProyectoError(\"La normalizacion dejo nombres duplicados.\")
    return normalizados


def seudonimizar(valor, largo=12):
    \"\"\"Devuelve un codigo seudonimo estable para un identificador personal.

    Se aplica a host_id (decision D5): deja contar anuncios por anfitrion
    sin guardar el identificador original. El mismo valor de entrada da
    siempre el mismo codigo, asi que el conteo sigue siendo valido.

    Ejemplo
    -------
    >>> seudonimizar(224592)  # doctest omite el valor exacto a proposito
    '...'
    \"\"\"
    if not isinstance(valor, (str, int)):
        raise ProyectoError(
            f\"Se esperaba texto o entero y se recibio {type(valor).__name__}.\")
    return hashlib.sha256(str(valor).encode(\"utf-8\")).hexdigest()[:largo]
"""

ARCHIVO_MODULO = DIR_SRC / "utilidades.py"
ARCHIVO_MODULO.write_text(CODIGO_MODULO, encoding="utf-8")
print("Módulo escrito en:", ARCHIVO_MODULO)

Módulo escrito en: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/utilidades.py


Para importarlo hay que agregar `src/` a la lista de rutas donde Python
busca módulos. `invalidate_caches()` hace falta porque el archivo se acaba de
crear: sin esa llamada, Python puede no ver un módulo escrito después de
arrancar el intérprete.

In [10]:
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))

importlib.invalidate_caches()
import utilidades                      # el módulo recién escrito
importlib.reload(utilidades)           # por si el cuaderno corre de nuevo

print("Importado:", utilidades.__name__, "desde", Path(utilidades.__file__).as_posix())
print(utilidades.normalizar_columnas(["Bathrooms Text", "Property Type", "Room Type"]))
print("host_id 224592 ->", utilidades.seudonimizar(224592))

Importado: utilidades desde /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/utilidades.py
['bathrooms_text', 'property_type', 'room_type']
host_id 224592 -> 8a3c202cc613


### 4.1 Pruebas del módulo (casos normal, límite y excepción)

Se aplican las mismas tres categorías que exige la rúbrica en la Fase 2.
`assert` frena el cuaderno si una condición no se cumple y deja constancia de
la falla en la salida ejecutada.

In [11]:
# --- Caso normal ---
assert utilidades.normalizar_nombre("Bathrooms Text") == "bathrooms_text"
assert utilidades.normalizar_columnas(["A B", "c-d"]) == ["a_b", "c_d"]
print("[OK] Caso normal: normalización correcta")

# --- Caso limite: signos seguidos y espacios en los bordes ---
assert utilidades.normalizar_nombre("  ¿Host ID (anfitrión)?  ") == "host_id_anfitri_n"
assert utilidades.normalizar_columnas([]) == []
print("[OK] Caso límite: signos y lista vacía")

# --- Seudonimización: determinista y distintiva ---
h1 = utilidades.seudonimizar(224592)
assert utilidades.seudonimizar(224592) == h1, "Mismo valor, distinto hash: falla la trazabilidad"
assert utilidades.seudonimizar(244792) != h1, "Dos anfitriones no deben colisionar"
assert len(h1) == 12 and "224592" not in h1, "El código no debe exponer el identificador"
print("[OK] Seudonimización: estable, distintiva y sin el valor original")

# --- Excepcion 1: tipo que no es ---
try:
    utilidades.normalizar_nombre(42)
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (tipo): {e}")

# --- Excepción 2: la normalización colapsa dos nombres en uno ---
# "Beds" y " beds " dan los dos "beds": el módulo lo detecta en vez de perder una columna.
try:
    utilidades.normalizar_columnas(["Beds", " beds "])
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (duplicados): {e}")

# --- Excepcion 3: seudonimizar un tipo no soportado ---
try:
    utilidades.seudonimizar(4.5)
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (seudónimo): {e}")

[OK] Caso normal: normalización correcta
[OK] Caso límite: signos y lista vacía
[OK] Seudonimización: estable, distintiva y sin el valor original
[OK] Excepción capturada (tipo): Se esperaba texto y se recibio int.
[OK] Excepción capturada (duplicados): La normalizacion dejo nombres duplicados.
[OK] Excepción capturada (seudónimo): Se esperaba texto o entero y se recibio float.


> **Por qué una excepción propia.** `ProyectoError` hereda de `Exception`
> y deja distinguir un fallo del proyecto de uno de pandas o de scikit-learn.
> Cuando el cuaderno de la Fase 3 capture errores, va a poder tratar distinto
> «los datos venían mal» y «mi función recibió algo que no esperaba».

## 5. Selección y ficha del conjunto de datos

**Qué hace este paso.** Documenta de dónde sale el conjunto y lo evalúa contra
los criterios de selección del curso.

**Por qué en la Fase 1.** Porque la elección del conjunto condiciona todo lo
que viene después. Un conjunto ya limpio convierte la sección de
preprocesamiento de la Fase 2 en un trámite; el de Inside Airbnb llega con el
precio como texto, una lista de comodidades en formato JSON y columnas
completamente vacías —y el archivo original trae además datos personales, que
la generación de la muestra elimina—: hay trabajo real que justificar.

### 5.1 Procedencia

| Campo | Valor |
| --- | --- |
| Título | Inside Airbnb — Santiago, Chile (`listings.csv` detallado) |
| Autor | Inside Airbnb |
| Plataforma | Inside Airbnb, sección *Get the data* |
| Enlace | `http://insideairbnb.com/get-the-data/` |
| Captura | 25–26 de septiembre de 2026 (`last_scraped`) |
| Estructura original | 18.904 filas × 90 columnas (~50 MB, contiene datos personales: no se versiona) |
| Archivo de trabajo | Muestra estratificada por comuna **anonimizada**: 5.106 filas × 78 columnas (`src/generar_muestra.py`, semilla 42) |
| Unidad de observación | Un alojamiento publicado en Airbnb Santiago |
| Licencia | Datos públicos con fines no comerciales, declarados en la plataforma |

**Referencia en APA 7 para el informe:**

> Inside Airbnb. (2026). *Get the data: Santiago, Chile* [Conjunto de datos]. http://insideairbnb.com/get-the-data/

Documentar la procedencia no es un trámite: sin ella nadie puede verificar
sobre qué versión del archivo se trabajó. La decisión de trabajar con una
muestra (D8) también queda declarada acá, con su mecanismo de reproducción. Y
la muestra **sí se puede versionar** porque la anonimización (D5) se aplica en
la misma generación: `src/generar_muestra.py` elimina las 12 columnas con datos
personales y seudonimiza `host_id`.

In [12]:
FICHA = {
    "titulo": "Inside Airbnb - Santiago, Chile (listings detallado, muestra estratificada)",
    "autor": "Inside Airbnb",
    "plataforma": "Inside Airbnb - Get the data",
    "url": "http://insideairbnb.com/get-the-data/",
    "licencia_abierta": True,
    "unidad_observacion": "Un alojamiento publicado",
    "filas": 5106,
    "columnas": 78,
    "tamano_mb": 8.8,
    # Conteo de variables por ROL ANALITICO (clasificar de validar_dataset.py),
    # no por tipo de dato: el rol es el que manda en el preprocesamiento.
    "roles": {
        "continua": 17,         # latitude, review_scores_*, estimated_revenue...
        "discreta": 27,         # accommodates, availability_*, number_of_reviews...
        "binaria": 5,           # host_is_superhost, has_availability...
        "nominal": 3,           # room_type, neighbourhood_cleansed, host_has_profile_pic
        "ordinal": 0,
        "fecha": 4,             # last_scraped, first_review, last_review, calendar_last_scraped
        "alta_cardinalidad": 8,  # amenities, property_type, host_id seudonimizado
        "identificador": 1,     # id
        "vacia": 13,            # columnas 100% nulas: se eliminan en la Fase 2
    },
    # % máximo de nulos entre variables NO vacías (license); y el máximo absoluto
    # (las 14 columnas vacías): el primero acredita faltantes reales y el
    # segundo deja el aviso que resuelve la Fase 2.
    "pct_nulos_max_no_vacia": 99.0,
    "pct_nulos_max_variable": 100.0,
}

### 5.2 Evaluación contra los criterios del curso

La función compara la ficha con los mínimos definidos para el proyecto y
devuelve una tabla con el veredicto por criterio. No decide por el grupo:
muestra lo que falta.

In [13]:
# Requisitos del curso. Los OBLIGATORIOS descalifican el conjunto si no se
# cumplen; los AVISOS no, pero lo que falte se declara en el informe como
# restricción del conjunto. Son los mismos que aplica validar_dataset.py.
OBLIGATORIOS = {
    "filas": ("Filas", 2000, "Permite agrupar por categoría sin grupos vacíos"),
    "columnas": ("Columnas", 12, "Asegura variedad de roles analíticos"),
    "roles": ("Roles analíticos distintos", 3, "El conjunto debe ser multivariado"),
    "numerica": ("Variables numéricas", 1, "Escalamiento y valores atípicos"),
    "categorica": ("Variables categóricas", 1, "Codificación de categorías"),
}
AVISOS = {
    "fecha": ("Fecha", 1, "Parseo y derivación de variables temporales"),
    "alta_cardinalidad": ("Texto o alta cardinalidad", 1, "Normalización y agrupación"),
}

# Roles que cuentan para cada grupo. El identificador y las vacías no son
# roles analíticos: el primero solo distingue filas y las otras no traen nada.
NUMERICOS = ("continua", "discreta")
CATEGORICOS = ("binaria", "nominal", "ordinal")
SIN_ROL = ("identificador", "vacia")


def evaluar_criterios(ficha, obligatorios=OBLIGATORIOS, avisos=AVISOS):
    """
    Compara una ficha de conjunto de datos con los requisitos del curso.

    Parametros
    ----------
    ficha : dict
        Ficha del conjunto, con las claves 'filas', 'columnas' y 'roles'.
    obligatorios, avisos : dict
        Criterio -> (etiqueta, minimo, motivo).

    Retorna
    -------
    pd.DataFrame
        Una fila por criterio, con su tipo, el valor observado y el veredicto.

    Lanza
    -----
    KeyError
        Si la ficha no declara los roles analiticos.
    """
    if "roles" not in ficha:
        raise KeyError("La ficha debe declarar el conteo de variables por rol analítico.")

    roles = ficha["roles"]
    observado = {
        "filas": ficha.get("filas", 0),
        "columnas": ficha.get("columnas", 0),
        # Cuántos roles distintos salen al menos una vez, sin contar los
        # que no aportan al análisis (identificador y vacías).
        "roles": sum(1 for rol, n in roles.items() if n > 0 and rol not in SIN_ROL),
        "numerica": sum(roles.get(r, 0) for r in NUMERICOS),
        "categorica": sum(roles.get(r, 0) for r in CATEGORICOS),
        "fecha": roles.get("fecha", 0),
        "alta_cardinalidad": roles.get("alta_cardinalidad", 0),
    }

    filas = []
    for tipo, criterios in (("obligatorio", obligatorios), ("aviso", avisos)):
        for clave, (etiqueta, minimo, motivo) in criterios.items():
            valor = observado[clave]
            # Un aviso incumplido no dice NO: dice que hay algo por declarar.
            falla = "NO" if tipo == "obligatorio" else "aviso"
            filas.append({
                "criterio": etiqueta,
                "tipo": tipo,
                "mínimo": minimo,
                "observado": valor,
                "cumple": "sí" if valor >= minimo else falla,
                "por qué se pide": motivo,
            })

    # Los faltantes se informan como aviso: un conjunto sin ellos sirve igual,
    # pero deja con poca evidencia la sección de preprocesamiento de la Fase 2.
    pct = ficha.get("pct_nulos_max_no_vacia", 0.0)
    filas.append({
        "criterio": "Alguna variable con ≥1% de nulos", "tipo": "aviso",
        "mínimo": 1.0, "observado": pct,
        "cumple": "sí" if pct >= 1.0 else "aviso",
        "por qué se pide": "Un conjunto perfecto no tiene preprocesamiento que justificar",
    })
    pct_max = ficha.get("pct_nulos_max_variable", 0.0)
    filas.append({
        "criterio": "Ninguna variable sobre 60% de nulos", "tipo": "aviso",
        "mínimo": 60.0, "observado": pct_max,
        "cumple": "sí" if pct_max <= 60.0 else "aviso",
        "por qué se pide": "Por encima de eso la variable no es recuperable",
    })
    tabla = pd.DataFrame(filas)
    for col in ("mínimo", "observado"):
        tabla[col] = tabla[col].map(lambda v: f"{v:g}")
    return tabla


evaluacion = evaluar_criterios(FICHA)
evaluacion

,criterio,tipo,mínimo,observado,cumple,por qué se pide
0,Filas,obligatorio,2000,5106,sí,Permite agrupar por categoría sin grupos vacíos
1,Columnas,obligatorio,12,78,sí,Asegura variedad de roles analíticos
2,Roles analíticos distintos,obligatorio,3,6,sí,El conjunto debe ser multivariado
3,Variables numéricas,obligatorio,1,44,sí,Escalamiento y valores atípicos
4,Variables categóricas,obligatorio,1,8,sí,Codificación de categorías
5,Fecha,aviso,1,4,sí,Parseo y derivación de variables temporales
6,Texto o alta cardinalidad,aviso,1,8,sí,Normalización y agrupación
7,Alguna variable con ≥1% de nulos,aviso,1,99,sí,Un conjunto perfecto no tiene preprocesamiento...
8,Ninguna variable sobre 60% de nulos,aviso,60,100,aviso,Por encima de eso la variable no es recuperable


In [14]:
incumplidos = evaluacion.loc[evaluacion["cumple"] == "NO", "criterio"].tolist()
avisos = evaluacion.loc[evaluacion["cumple"] == "aviso", "criterio"].tolist()
print(f"Criterios evaluados : {len(evaluacion)}")
print(f"Obligatorios sin cumplir: {len(incumplidos)}")
print(f"Avisos              : {len(avisos)}")
if incumplidos:
    print("\nNo cumple (descalifica el conjunto):")
    for c in incumplidos:
        print("  ·", c)
if avisos:
    print("\nAvisos (no descalifican; se declaran en el informe):")
    for c in avisos:
        print("  ·", c)

Criterios evaluados : 9
Obligatorios sin cumplir: 0
Avisos              : 1

Avisos (no descalifican; se declaran en el informe):
  · Ninguna variable sobre 60% de nulos


> **Léase con atención el resultado.** El conjunto cumple los cinco
> requisitos obligatorios y deja un aviso importante: **hay variables con más
> del 60 % de faltantes** —13 columnas están 100 % vacías y `license` llega al
> 99 %—, y `source` pasa del 95 %. No descalifica: es justo el tipo de
> problema que la Fase 2 tiene que resolver y justificar (esas columnas se
> eliminan porque no traen información). El resultado se confirma sobre el
> archivo real con `validar_dataset.py`, cuyo informe queda en
> `docs/validacion_dataset.md`.

### 5.3 Diccionario de variables

El diccionario declara el **rol analítico** de cada variable. Esa es la
decisión que determina todo el preprocesamiento de la Fase 2: dos columnas
`int64` pueden ser un identificador, una variable discreta o una binaria, y
cada una pide un tratamiento distinto.

El archivo tiene 78 columnas; el diccionario que sigue declara **las variables
que entran al análisis** (las que responden las tres preguntas y definen el
universo). La versión completa, con el rol observado de las 78 columnas, la
genera el cuaderno de la Fase 2 en `docs/diccionario_variables.csv`.

In [15]:
DICCIONARIO = [
    ("id",                        "identificador", "Identificador del alojamiento; une con reseñas"),
    ("host_id",                   "identificador", "Identificador del anfitrión; se seudonimiza (D5)"),
    ("neighbourhood_cleansed",    "nominal",       "Comuna (32 categorías)"),
    ("latitude",                  "continua",      "Latitud del alojamiento"),
    ("longitude",                 "continua",      "Longitud del alojamiento"),
    ("property_type",             "alta cardinalidad", "Tipo de propiedad (70 categorías; se agrupan en F2)"),
    ("room_type",                 "nominal",       "Entire home/apt, Private room, Hotel room, Shared room"),
    ("accommodates",              "discreta",      "Capacidad de huéspedes"),
    ("bedrooms",                  "discreta",      "Número de dormitorios"),
    ("beds",                      "discreta",      "Número de camas"),
    ("bathrooms_text",            "nominal",       "Baños en texto: '1 bath', '1.5 shared baths'..."),
    ("amenities",                 "texto libre",   "Lista JSON de comodidades; se binariza en F2"),
    ("price",                     "continua",      "Precio por noche; viene como texto '$59,303.00'"),
    ("minimum_nights",            "discreta",      "Noches mínimas de estadía"),
    ("availability_365",          "discreta",      "Días disponibles en el próximo año; define 'activo'"),
    ("number_of_reviews",         "discreta",      "Reseñas acumuladas"),
    ("number_of_reviews_ltm",     "discreta",      "Reseñas últimos 12 meses; define 'activo'"),
    ("number_of_reviews_l30d",    "discreta",      "Reseñas últimos 30 días"),
    ("reviews_per_month",         "continua",      "Reseñas por mes; nula cuando no hay reseñas"),
    ("estimated_occupancy_l365d", "discreta",      "Ocupación estimada últimos 365 días; tope 255 noches"),
    ("estimated_revenue_l365d",   "continua",      "Ingresos estimados = ocupación × precio (derivada)"),
    ("first_review",              "fecha",         "Fecha de la primera reseña"),
    ("last_review",               "fecha",         "Fecha de la última reseña"),
    ("review_scores_rating",      "continua",      "Puntaje global; nulo cuando no hay reseñas"),
    ("host_is_superhost",         "binaria",       "t/f: anfitrión con distinción superhost"),
    ("calculated_host_listings_count", "discreta", "Anuncios del mismo anfitrión"),
]

diccionario = pd.DataFrame(DICCIONARIO,
                           columns=["variable", "rol_analitico", "descripcion"])

# Verificación: las variables declaradas tienen que existir en el archivo real.
assert diccionario["rol_analitico"].str.strip().ne("").all(), "Hay roles vacíos."
print(f"{len(diccionario)} variables declaradas de las {FICHA['columnas']} del archivo.")
print("(El rol observado de las 90 columnas se documenta en la Fase 2.)\n")
print(diccionario["rol_analitico"].value_counts().to_string())
diccionario

26 variables declaradas de las 78 del archivo.
(El rol observado de las 90 columnas se documenta en la Fase 2.)

rol_analitico
discreta             10
continua              6
nominal               3
identificador         2
fecha                 2
alta cardinalidad     1
texto libre           1
binaria               1


,variable,rol_analitico,descripcion
0,id,identificador,Identificador del alojamiento; une con reseñas
1,host_id,identificador,Identificador del anfitrión; se seudonimiza (D5)
2,neighbourhood_cleansed,nominal,Comuna (32 categorías)
3,latitude,continua,Latitud del alojamiento
4,longitude,continua,Longitud del alojamiento
5,property_type,alta cardinalidad,Tipo de propiedad (70 categorías; se agrupan e...
6,room_type,nominal,"Entire home/apt, Private room, Hotel room, Sha..."
7,accommodates,discreta,Capacidad de huéspedes
8,bedrooms,discreta,Número de dormitorios
9,beds,discreta,Número de camas


> **Tres avisos del diccionario.** (1) `price` se declara *continua* aunque
> el archivo la trae como texto (`$59,303.00`): el rol es analítico y la
> conversión es trabajo de la Fase 2. (2) `estimated_occupancy_l365d` y
> `estimated_revenue_l365d` **no salen en el diccionario oficial** de Inside
> Airbnb (v4.3, agosto 2022): el archivo trae 15 columnas no documentadas y la
> ocupación tiene un tope de 255 noches que se declara como sesgo. (3)
> `bathrooms_text` reemplaza a `bathrooms`, que llega 100 % vacía.

### 5.4 Comprobación del archivo (si ya está descargado)

Esta celda no descarga nada. Mira si la muestra está en `data/raw` y, si está,
contrasta la estructura real con lo declarado en la ficha y verifica que las
variables del diccionario existan. Si aún no se genera, avisa qué falta y el
cuaderno sigue corriendo igual: la Fase 1 no depende del archivo.

In [16]:
ARCHIVO = DIR_CRUDO / "listings.csv"

if ARCHIVO.exists():
    df_crudo = pd.read_csv(ARCHIVO)
    print("Archivo encontrado :", ARCHIVO.as_posix())
    print("Dimensiones reales :", df_crudo.shape)
    print("Declarado en ficha :", (FICHA["filas"], FICHA["columnas"]))
    coincide = df_crudo.shape == (FICHA["filas"], FICHA["columnas"])
    print("Coincidencia       :", "[OK]" if coincide else "[REVISAR] ficha y archivo difieren")

    # Las variables del diccionario tienen que estar en el archivo.
    ausentes = [v for v in diccionario["variable"] if v not in df_crudo.columns]
    print("Variables del diccionario ausentes en el archivo:",
          ausentes if ausentes else "ninguna [OK]")

    print("\nNulos relevantes (columnas con entre 1 y 99 % de faltantes):")
    nulos = (df_crudo.isna().mean() * 100).round(1)
    print(nulos[(nulos > 0) & (nulos < 99)].sort_values(ascending=False).to_string())
    print(f"\nColumnas 100 % vacías: {int((df_crudo.isna().mean() == 1).sum())}")
else:
    df_crudo = None
    print("[PENDIENTE] El archivo aún no está en", DIR_CRUDO.as_posix())
    print("1. Descargue 'listings.csv' (detallado) desde:", FICHA["url"])
    print("2. Genere la muestra:  python src/generar_muestra.py ruta/al/listings.csv")
    print("\nLa Fase 1 no requiere el archivo: la exploración es trabajo de la Fase 2.")

Archivo encontrado : /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/data/raw/listings.csv
Dimensiones reales : (5106, 78)
Declarado en ficha : (5106, 78)
Coincidencia       : [OK]
Variables del diccionario ausentes en el archivo: ninguna [OK]

Nulos relevantes (columnas con entre 1 y 99 % de faltantes):
source                         95.3
review_scores_communication    14.7
review_scores_location         14.7
last_review                    14.7
first_review                   14.7
review_scores_value            14.7
reviews_per_month              14.7
review_scores_accuracy         14.7
review_scores_rating           14.7
review_scores_checkin          14.7
review_scores_cleanliness      14.7
price_quote_price_per_night     4.7
price                           4.7
estimated_revenue_l365d         4.7
price_quote_total_price         4.7
price_quote_checkin_date        3.2
price_quote_checkout_date       3.2
price_quote_raw                 3.2
beds                            1.1
has_

## 6. Control de versiones

**Qué hace este paso.** Consulta el estado de Git desde el mismo cuaderno: si
está instalado, si la identidad está configurada, si la carpeta es un
repositorio y qué *commits* tiene.

**Por qué se consulta y no se ejecuta.** Este cuaderno **no** hace `git init`,
`add` ni `commit`. Versionar es una decisión de quien trabaja, y automatizarla
desde un cuaderno produce historiales sin sentido. Lo que si corresponde es
dejar constancia verificable del estado del repositorio al momento de la
entrega.

In [17]:
def git(*argumentos):
    """
    Ejecuta un comando de Git y devuelve su salida como texto.

    Retorna
    -------
    (bool, str)
        Exito de la ejecucion y salida (o mensaje de error).
    """
    if shutil.which("git") is None:
        return False, "Git no está instalado o no está en el PATH."
    try:
        proceso = subprocess.run(["git", *argumentos], capture_output=True,
                                 text=True, timeout=20, cwd=RAIZ)
    except (OSError, subprocess.SubprocessError) as e:
        return False, f"No se pudo ejecutar Git: {e}"
    if proceso.returncode != 0:
        # stderr trae el motivo real; se devuelve tal cual para no esconderlo.
        return False, proceso.stderr.strip() or "Git terminó con error."
    return True, proceso.stdout.strip()


ok, salida = git("--version")
print("Git                :", salida if ok else f"[AVISO] {salida}")

for clave in ("user.name", "user.email"):
    ok_cfg, valor = git("config", "--get", clave)
    estado = valor if ok_cfg and valor else "[FALTA] configúrelo con git config --global"
    print(f"{clave:19}:", estado)

Git                : git version 2.53.0
user.name          : Mauricio Martinez
user.email         : mmartinez@whitestack.com


> **Atención.** El correo configurado en Git tiene que ser **el mismo** de
> la cuenta de GitHub. Si no coinciden, los *commits* aparecen en el historial
> sin vincularse al perfil y en la revisión no se puede atribuir el trabajo a
> su autor. Corregirlo después implica reescribir el historial, que es un
> procedimiento delicado.

In [18]:
es_repo, _ = git("rev-parse", "--is-inside-work-tree")

if es_repo == "true" or es_repo is True:
    ok_rama, rama = git("rev-parse", "--abbrev-ref", "HEAD")
    print("Rama actual        :", rama if ok_rama else "sin commits todavía")

    ok_log, log = git("log", "--oneline", "-5")
    print("\nÚltimos commits")
    print(log if ok_log and log else "  (todavía no hay commits)")

    # --porcelain da una salida estable, pensada para que la lea un programa.
    ok_st, estado = git("status", "--porcelain")
    pendientes = [l for l in estado.splitlines() if l.strip()] if ok_st else []
    print(f"\nArchivos con cambios sin registrar: {len(pendientes)}")
    for linea in pendientes[:10]:
        print("  ", linea)

    ok_rem, remoto = git("remote", "-v")
    print("\nRemoto             :", remoto.splitlines()[0] if ok_rem and remoto
          else "[FALTA] agregue el remoto con git remote add origin URL")
else:
    print("[PENDIENTE] Esta carpeta todavía no es un repositorio Git.")
    print("Desde la terminal, en la raíz del proyecto:")
    print("  git init")
    print("  git add .gitignore requirements.txt README.md")
    print('  git commit -m "docs: estructura inicial del proyecto"')

Rama actual        : main

Últimos commits
79e9bcb docs: actualiza README, registro de decisiones y validacion del dataset
4323655 feat: anonimiza la muestra al generarla y verifica la regla en F2
7087905 data: versiona la muestra raw anonimizada sin datos personales
513146e fix: convierte price a numérico y filtra alojamientos activos
00335c1 feat: pipeline F2 de carga y anonimización de listings y reviews

Archivos con cambios sin registrar: 10
   M .gitignore
    M F1/S1_F1_Definicion.ipynb
    M F1/notebooks/proyeco.ipynb
    M F2/S1_F2_Preprocesamiento.ipynb
    M README.md
    M docs/metadatos_fase1.json
    M docs/registro_decisiones.md
    M src/generar_muestra.py
    M src/utilidades.py
    M validar_dataset.py

Remoto             : origin	git@github.com:RedWolfo10/proyecto-grupo4-mcdi500.git (fetch)


**Convención de *commits* del curso.** Los prefijos `docs`, `data`, `feat`
y `fix` ordenan el historial y permiten leerlo como una bitácora. Un *commit*
por bloque de trabajo con sentido propio, no uno al final del día.

| Prefijo | Cuándo | Ejemplo |
| --- | --- | --- |
| `docs` | Documentación, README, informe | `docs: agrega registro de decisiones` |
| `data` | Incorporación o actualización de datos | `data: agrega salida procesada del pipeline` |
| `feat` | Código nuevo que aporta funcionalidad | `feat: agrega notebook F1 con validacion del dataset` |
| `fix` | Corrección de un defecto | `fix: convierte price a numerico` |

> **Atención.** Mejor evitar tildes y caracteres especiales en los mensajes de
> *commit*: según la configuración de cada terminal, salen corruptos en el
> historial compartido.

## 7. Validación técnica de la fase

Verificar es demostrar, con evidencia, que el resultado es correcto. La
función `validar_fase1` reúne las comprobaciones de la fase y usa `assert`: si
algo no se cumple, el cuaderno se detiene con un error y queda constancia de
la falla.

Se comprueba:
1. **Estructura** — que existan las carpetas comprometidas.
2. **Artefactos** — que `.gitignore` y `requirements.txt` estén escritos y no vacíos.
3. **Definición** — que la configuración del proyecto tenga sus claves obligatorias.
4. **Coherencia** — que el diccionario declare roles y sus variables existan en el archivo.
5. **Módulo** — que `src/utilidades.py` se pueda importar.

In [19]:
def validar_fase1(raiz, config, ficha, diccionario, df_archivo=None):
    """
    Comprueba que la Fase 1 dejo lo que la rubrica exige.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DE LA FASE 1")
    print("-" * 45)

    # 1. Estructura de carpetas
    esperadas = ["data/raw", "data/processed", "docs", "src", "F1", "F2", "F3", "F4"]
    faltan = [c for c in esperadas if not (raiz / c).is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura completa ({len(esperadas)} carpetas)")

    # 2. Artefactos del entorno reproducible
    for nombre in (".gitignore", "requirements.txt"):
        archivo = raiz / nombre
        assert archivo.exists(), f"Falta {nombre}."
        assert archivo.stat().st_size > 0, f"{nombre} está vacío."
        print(f"[OK] {nombre} presente ({archivo.stat().st_size} bytes)")

    # 3. Definición del problema
    for clave in ("titulo", "problematica", "objetivo_general", "objetivos_especificos"):
        assert config.get(clave), f"La configuración no declara '{clave}'."
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos."
    print("[OK] Definición del problema completa")

    # 4. Coherencia del diccionario: roles declarados y variables existentes
    assert len(diccionario) >= 15, "El diccionario debe cubrir las variables del análisis."
    roles_declarados = set(diccionario["rol_analitico"])
    assert roles_declarados, "El diccionario no declara roles analíticos."
    if df_archivo is not None:
        ausentes = set(diccionario["variable"]) - set(df_archivo.columns)
        assert not ausentes, f"Variables del diccionario ausentes del archivo: {ausentes}"
        print(f"[OK] Diccionario coherente ({len(diccionario)} variables declaradas "
              f"de {df_archivo.shape[1]}, {len(roles_declarados)} roles)")
    else:
        print(f"[OK] Diccionario declarado ({len(diccionario)} variables, "
              f"{len(roles_declarados)} roles; archivo pendiente)")

    # 5. Módulo importable
    assert (raiz / "src" / "utilidades.py").exists(), "Falta el módulo src/utilidades.py."
    assert callable(getattr(utilidades, "normalizar_columnas", None)), \
        "El módulo no expone normalizar_columnas."
    assert callable(getattr(utilidades, "seudonimizar", None)), \
        "El módulo no expone seudonimizar."
    print("[OK] Módulo src/utilidades.py importable")

    print(f"\n[INFO] Conjunto declarado: {ficha['titulo']}")
    print(f"[INFO] Archivo generado: {'sí' if ARCHIVO.exists() else 'todavía no'}")
    return True


validar_fase1(RAIZ, PROYECTO, FICHA, diccionario, df_crudo)

VALIDACION DE LA FASE 1
---------------------------------------------
[OK] Estructura completa (8 carpetas)
[OK] .gitignore presente (493 bytes)
[OK] requirements.txt presente (112 bytes)
[OK] Definición del problema completa
[OK] Diccionario coherente (26 variables declaradas de 78, 8 roles)
[OK] Módulo src/utilidades.py importable

[INFO] Conjunto declarado: Inside Airbnb - Santiago, Chile (listings detallado, muestra estratificada)
[INFO] Archivo generado: sí


True

## 8. Vinculación con el mapa conceptual

La rúbrica de la Sumativa 1 evalúa por separado que el mapa conceptual de la
Formativa 1 corresponda con lo que efectivamente se construyó. La tabla de
abajo hace explícita esa correspondencia: cada nodo del mapa, dónde está
implementado y qué evidencia lo respalda.

In [20]:
VINCULACION = [
    # (nodo del mapa conceptual, dónde se implementa, evidencia verificable)
    ("Entorno de desarrollo en Python",      "Sección 2 · verificar_entorno",     "Intérprete .venv y versiones impresas"),
    ("Ambiente virtual (.venv)",             "Sección 2 · verificar_entorno",     "Ruta del intérprete contiene .venv"),
    ("requirements.txt",                     "Sección 3.2 · escribir_requirements", "Archivo generado desde el entorno"),
    ("Ambiente virtual registra dependencias","Sección 3.2 · requirements.txt",    "Dependencias con versión fija"),
    ("Git y GitHub",                         "Sección 6 · consultas a Git",        "Rama, commits, remoto y correo"),
    ("Commits descriptivos y ramas",         "Sección 6 · convención de commits",  "Historial con prefijos docs/feat/fix"),
    ("Repositorio del proyecto",             "Sección 3 · localizar_raiz y árbol", "Estructura impresa y validada"),
    ("Carpetas F1 y F2",                     "Sección 3 · DIR_FASES",              "Carpetas F1–F4 creadas"),
    ("Jupyter Notebook",                     "Este cuaderno",                      "F1/S1_F1_Definicion.ipynb ejecutado"),
    ("NumPy y Pandas",                       "Secciones 5 y 9 · tablas de documentación", "Versiones verificadas y DataFrames"),
    ("Funciones, módulos y scripts reutilizables", "Sección 4 · src/utilidades.py", "Módulo importado y probado"),
    ("Documentación en Markdown y comentarios", "Celdas narrativas y docstrings",  "Cuaderno legible, docs/ generado"),
    ("README del proyecto",                  "Sección 9 · generar_readme",         "README.md en la raíz"),
    ("Validación técnica y pruebas",         "Secciones 4.1 y 7 · asserts",        "Casos normal, límite y excepciones"),
    ("Datos públicos de Inside Airbnb",      "Sección 5 · FICHA y procedencia",    "Ficha, cita APA 7 y evaluación"),
    ("Preparación y análisis de los datos",  "Muestra estratificada + Fase 2",     "src/generar_muestra.py y F2/"),
]

vinculacion = pd.DataFrame(VINCULACION,
                           columns=["nodo_mapa", "donde_se_implementa", "evidencia"])

# Ningún nodo del mapa puede quedar sin evidencia: eso es lo que revisa la rúbrica.
assert vinculacion["evidencia"].str.strip().ne("").all(), "Hay nodos sin evidencia."
print(f"{len(vinculacion)} nodos del mapa conceptual, todos con evidencia asociada.\n")
vinculacion

16 nodos del mapa conceptual, todos con evidencia asociada.



,nodo_mapa,donde_se_implementa,evidencia
0,Entorno de desarrollo en Python,Sección 2 · verificar_entorno,Intérprete .venv y versiones impresas
1,Ambiente virtual (.venv),Sección 2 · verificar_entorno,Ruta del intérprete contiene .venv
2,requirements.txt,Sección 3.2 · escribir_requirements,Archivo generado desde el entorno
3,Ambiente virtual registra dependencias,Sección 3.2 · requirements.txt,Dependencias con versión fija
4,Git y GitHub,Sección 6 · consultas a Git,"Rama, commits, remoto y correo"
5,Commits descriptivos y ramas,Sección 6 · convención de commits,Historial con prefijos docs/feat/fix
6,Repositorio del proyecto,Sección 3 · localizar_raiz y árbol,Estructura impresa y validada
7,Carpetas F1 y F2,Sección 3 · DIR_FASES,Carpetas F1–F4 creadas
8,Jupyter Notebook,Este cuaderno,F1/S1_F1_Definicion.ipynb ejecutado
9,NumPy y Pandas,Secciones 5 y 9 · tablas de documentación,Versiones verificadas y DataFrames


> **Cómo se escribe esto en el informe.** La tabla sola no basta. Cada
> fila se tiene que poder señalar en el repositorio: un archivo, una celda o
> un *commit*. Un nodo del mapa que no tiene dónde apuntar es un nodo que se
> dibujó pero no se construyó.

## 9. Persistencia y trazabilidad

El resultado tiene que quedar guardado y documentado. Esta sección produce
los artefactos que se versionan en el repositorio y se citan en el informe
técnico.

In [21]:
def generar_readme(config, ficha, requisitos, ruta):
    """Escribe el README tecnico del repositorio a partir de la configuracion."""
    integrantes = "\n".join(f"- {n} (@usuario-github)" for n in config["integrantes"])
    dependencias = "\n".join(f"- {r}" for r in requisitos)

    contenido = f"""# {config['titulo']} — {config['grupo']}

{config['problematica']}

## Integrantes
{integrantes}

## Datos
- Fuente: {ficha['titulo']} ({ficha['autor']}, {ficha['plataforma']})
- Enlace: {ficha['url']}
- Estructura original: 18.904 filas x 90 columnas; archivo de trabajo:
  muestra estratificada y anonimizada de {ficha['filas']} filas x {ficha['columnas']} columnas
  (semilla 42, mínimo un registro por comuna)
- Ubicación esperada: `data/raw/listings.csv` (versionada: ya viene anonimizada)

### Cómo obtener los datos
1. Descargar `listings.csv` (versión detallada) de Inside Airbnb, Santiago.
   La descarga original trae datos personales y NO se versiona (`*.csv.gz`).
2. Generar la muestra en la raíz del proyecto:

       python src/generar_muestra.py ruta/al/listings.csv

   El script es determinista: con la misma entrada da la misma muestra,
   ya anonimizada (sin nombres, enlaces ni fotos; `host_id` con hash).

## Estructura del repositorio
```
data/raw/        muestra anonimizada generada por src/generar_muestra.py
data/processed/  datos tras limpieza y transformación (F2)
docs/            diccionario, registro de decisiones, metadatos y validación
src/             módulos reutilizables del proyecto
F1/ F2/ F3/ F4/  cuadernos e informes de cada fase
```

## Requisitos y ejecución
Python 3.13 o superior

    python -m venv .venv
    source .venv/Scripts/activate    # Windows, Git Bash
    # source .venv/bin/activate      # macOS y Linux
    python -m pip install -r requirements.txt

Correr los cuadernos en orden, primero F1 y después F2. Cada cuaderno
encuentra la raíz del proyecto aunque Jupyter lo abra en su propia carpeta.

### Dependencias declaradas
{dependencias}

## Convención de commits
Prefijos usados: docs, data, feat, fix.

## Decisiones técnicas
- Los datos son comunes a todas las fases (`data/`); los cuadernos se separan por fase.
- Semilla aleatoria fijada en {SEMILLA} para asegurar reproducibilidad.
- `data/raw/` sí se versiona porque la muestra llega anonimizada; las
  descargas originales (`*.csv.gz`) no (datos personales y peso).
- Registro de decisiones numeradas (D1–D8) en `docs/registro_decisiones.md`.
"""
    ruta.write_text(contenido, encoding="utf-8")
    return contenido


ARCHIVO_README = RAIZ / "README.md"
contenido = generar_readme(PROYECTO, FICHA, requisitos, ARCHIVO_README)
print(f"{ARCHIVO_README} escrito ({len(contenido.splitlines())} líneas)\n")
print("\n".join(contenido.splitlines()[:14]), "\n...")

/home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/README.md escrito (66 líneas)

# Análisis de alojamientos Airbnb en Santiago de Chile para la decisión de inversión inmobiliaria — Grupo 4

Un inversionista que está evaluando comprar un departamento para arriendo turístico en Santiago no tiene evidencia sistemática sobre qué ubicación, tipología y equipamiento se asocian a mayores ingresos. El proyecto describe esa relación a partir de 18.904 alojamientos publicados en Airbnb (Inside Airbnb, captura del 25–26 de septiembre de 2026), de los que se toma una muestra estratificada por comuna.

## Integrantes
- Alfredo Valenzuela Aburto (@usuario-github)
- Mauricio Martínez Quiñones (@usuario-github)
- María Laura Lanosa (@usuario-github)

## Datos
- Fuente: Inside Airbnb - Santiago, Chile (listings detallado, muestra estratificada) (Inside Airbnb, Inside Airbnb - Get the data)
- Enlace: http://insideairbnb.com/get-the-data/
- Estructura original: 18.904 filas x 90 columnas; archivo de 

In [22]:
# Artefactos de documentación de la fase, en formato de tabla para el informe.
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
evaluacion.to_csv(DIR_DOCS / "evaluacion_criterios_dataset.csv", index=False)
vinculacion.to_csv(DIR_DOCS / "vinculacion_mapa_conceptual.csv", index=False)

# Metadatos de la fase: lo que deja reconstruir en qué condiciones corrió.
METADATOS = {
    "proyecto": PROYECTO["titulo"],
    "grupo": PROYECTO["grupo"],
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "semilla": SEMILLA,
    "python": sys.version.split()[0],
    "sistema": f"{platform.system()} {platform.release()}",
    "interprete": ENTORNO["interprete"],
    "entorno_virtual": ENTORNO["entorno_virtual"],
    "dependencias": ENTORNO["versiones"],
    "dataset": {"titulo": FICHA["titulo"], "url": FICHA["url"],
                "archivo_generado": ARCHIVO.exists(),
                "filas_muestra": FICHA["filas"]},
}
(DIR_DOCS / "metadatos_fase1.json").write_text(
    json.dumps(METADATOS, indent=2, ensure_ascii=False), encoding="utf-8")

for archivo in sorted(DIR_DOCS.iterdir()):
    print(f"  {archivo.as_posix():48} {archivo.stat().st_size:6} bytes")

  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/diccionario_variables.csv   1666 bytes
  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/evaluacion_criterios_dataset.csv    785 bytes
  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/metadatos_fase1.json    754 bytes
  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/metadatos_proyecto.csv   1120 bytes
  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/registro_decisiones.md   3480 bytes
  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/validacion_dataset.md   5353 bytes
  /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/docs/vinculacion_mapa_conceptual.csv   1508 bytes


In [23]:
# Resumen de cierre: qué había al empezar y qué queda al terminar la fase.
resumen = pd.DataFrame([
    {"artefacto": "Carpetas del proyecto",
     "estado": len([p for p in RAIZ.rglob("*") if p.is_dir()
                    and not any(parte.startswith(".") for parte in p.parts)])},
    {"artefacto": "Archivos en docs/", "estado": len(list(DIR_DOCS.iterdir()))},
    {"artefacto": "Dependencias declaradas", "estado": len(requisitos)},
    {"artefacto": "Variables en el diccionario", "estado": len(diccionario)},
    {"artefacto": "Nodos del mapa vinculados", "estado": len(vinculacion)},
    {"artefacto": "Criterios de dataset incumplidos", "estado": len(incumplidos)},
])
resumen

,artefacto,estado
0,Carpetas del proyecto,12
1,Archivos en docs/,7
2,Dependencias declaradas,8
3,Variables en el diccionario,26
4,Nodos del mapa vinculados,16
5,Criterios de dataset incumplidos,0


## Conclusiones y trazabilidad

La fase deja funcionando un entorno verificado, una estructura de repositorio
con sus artefactos de reproducibilidad (`.gitignore`, `requirements.txt`,
`README.md`), un módulo propio importable con las utilidades que la Fase 2
reutiliza, y la documentación del conjunto de datos con su evaluación frente
a los criterios del curso. Cada paso se implementó como una función documentada
y verificada (`presentar_proyecto`, `verificar_entorno`, `escribir_requirements`,
`evaluar_criterios`, `git`, `validar_fase1`, `generar_readme`).

**Trazabilidad con el repositorio (F1):** este cuaderno está en la carpeta
`F1/`; el `README` documenta dependencias, descarga de datos e instrucciones
de ejecución; los archivos de `docs/` sostienen las tablas del informe técnico
y el historial de *commits* refleja el avance descrito en él.

---

## 10. Reflexión técnica

### Hallazgos

- **El archivo llega con problemas reales que justifican la Fase 2.** De las
  90 columnas, 14 están 100 % vacías, `license` pasa del 99 % de faltantes, el
  precio viene como texto (`$59,303.00`) y las comodidades llegan como una
  lista JSON de 1.793 valores distintos. Nada de esto se notaba sin perfilar
  el archivo.

- **El diccionario oficial está desactualizado.** El conjunto trae 15 columnas
  no documentadas en la versión 4.3 (agosto 2022), entre ellas
  `estimated_occupancy_l365d` y `estimated_revenue_l365d`, que son justo las
  métricas de desempeño del proyecto. Verificar la fuente en vez de confiar en
  su documentación nos cambió la lectura del conjunto.

- **Los ingresos estimados no son una métrica independiente.** En el 100 % de
  los casos `estimated_revenue_l365d` equivale a ocupación × precio publicado.
  Por eso la métrica principal es la ocupación estimada y los ingresos quedan
  como derivada (decisión D3): usarlos como si fueran tres medidas distintas
  habría inflado las conclusiones.

- **El rol analítico, y no el tipo de dato, es el que define el trabajo
  pendiente.** `price` es texto pero su rol es continuo; `id` y `host_id` son
  enteros pero son identificadores; `amenities` es texto pero se comporta como
  categoría múltiple.

### Dificultades y cómo se resolvieron

- **El archivo detallado pesa ~50 MB y trae datos personales.** Se resolvió
  con dos decisiones documentadas: la descarga original queda por fuera de Git
  y los cuadernos trabajan con una muestra estratificada por comuna (D8) que
  conserva las 32 comunas. Como `src/generar_muestra.py` la anonimiza al
  generarla (D5), la muestra sí se puede versionar en `data/raw/` (D7).

- **Rutas.** Usamos rutas relativas con `pathlib` y una sola convención de
  carpetas de datos, compartida con el cuaderno de la Fase 2.

### Decisiones que quedan abiertas

- Si inferir el idioma de las reseñas (`reviews.csv.gz`, 739.949 registros)
  justifica el costo como aproximación a la demanda extranjera (D6, se evalúa
  en F3).
- Si la muestra del 27 % alcanza o conviene subir la fracción en las fases
  finales.
- Si `property_type` conviene agruparla en F2 o analizarla tal cual.

> **Declarar lo que queda abierto evidencia mas dominio que presentar todo
> resuelto.** Y en esta fase no debería estarlo.

---

## 11. Verificación antes de entregar

**Cuaderno**

- Corre completo tras *Kernel → Restart Kernel and Run All Cells*, sin errores.
- La numeración de ejecución queda continua.
- Cada bloque de código va precedido de una celda narrativa que explica su lógica.
- Todas las rutas son relativas y la semilla quedó declarada.

**Contenido técnico**

- La problemática, los objetivos, el alcance y los sesgos están declarados.
- El entorno se verifica con evidencia (intérprete, entorno virtual, versiones).
- La procedencia del conjunto está documentada, con enlace y cita en APA 7.
- El diccionario declara el rol analítico de las variables del análisis.
- El conjunto candidato se evaluó contra los criterios del curso.
- Hay pruebas de casos normales, límite y excepciones.

**Archivos**

- `.gitignore` y `requirements.txt` en la raíz, los dos con contenido.
- `README.md` técnico, generado desde la configuración del proyecto.
- `docs/` con diccionario, evaluación del conjunto, vinculación, metadatos y
  registro de decisiones.
- `src/utilidades.py` y `src/generar_muestra.py` disponibles.

**Repositorio**

- Carpeta F1 con este cuaderno, *commits* de todos los integrantes y `README`
  actualizado.
- El correo de Git coincide con el de la cuenta de GitHub.

---

*Cuaderno de la Fase 1 · MCDI500 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*